# Semana del 8: caso base replicando a Nano

**Por qué este run.** El repo de Nano tiene una sola curva de la galleta: visión completa y la galleta cambiando de esquina cada vez que se aprieta el botón (nuestro `cookie_full`). Ese run consiguió **0 galletas hasta los ~620.000 pasos**, y a los ~700.000 ya sacaba **~21 por partida**. Nuestros casos base anteriores se cortaron a los 100.000 y 300.000 pasos, antes de ese punto. Este notebook repite su configuración con nuestro código.

| | Casos base anteriores | Este run (como Nano) |
|---|---|---|
| Tarea | `cookie_fullfixed` | `cookie_full` |
| Pasos | 300.000 | 1.000.000 |
| `TRAIN_RATIO` | 128 | 256 (una actualización cada 4 pasos) |
| Imagen | 64×64 | 80×80 |
| Buffer | 100.000, en la GPU | 1.000.000 (todo el run), en la RAM |
| Sampler de galletas | 2 secuencias por batch | No (Nano no lo usó) |
| Evaluación | cada 10.000 pasos, 10 partidas | cada 50.000, 5 partidas |

Se mantienen el decoder (`REP_LOSS = "dreamer"`), el caso base y las 3 acciones (Nano usaba 7, así que para nosotros apretar el botón es más fácil).

**`cookie_full` y `cookie_fullfixed`** tienen visión completa y en las dos la galleta aparece solo al apretar el botón. La diferencia es qué pasa si el agente aprieta el botón otra vez con la galleta todavía en el mapa: en `cookie_full` la galleta se cambia a otra esquina al azar; en `cookie_fullfixed` no pasa nada. `cookie_full` es la que tiene curva de referencia y, con visión parcial, es el caso del profe (`cookie_partial`).

**Qué esperar.** Hasta los ~600.000 pasos lo normal es no ver galletas. El progreso previo se ve en las métricas nuevas (veces que aprieta el botón, casillas visitadas, pasos en el cuarto del botón) y en los videos de TensorBoard. Con una L4 son unas 25 horas (~40 unidades), repartidas en varias sesiones.

**Primera vez:** celdas 1 a 11, en orden. **Para seguir otro día:** celdas 1 a 6, la 7 si quieres ver en qué va, la 9 para tener TensorBoard, y la 10. Para mirar el progreso mientras entrena, abre `monitor.ipynb` en otra pestaña.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **L4 GPU**.

## 1. Revisar la GPU y la RAM

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os

ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM: {ram:.0f} GB")
if ram < 45:
    print("OJO: el buffer de este run ocupa ~30 GB de RAM. Usa la L4 (viene con 53 GB) o activa High-RAM.")

## 2. Montar Drive

Aquí quedan el checkpoint, el buffer y las métricas, para que sobrevivan al corte de sesión.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clonar el repo (o actualizarlo si ya está)

In [ ]:
%%bash
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi

## 4. Instalar

r2dreamer necesita Python 3.11 y Colab suele traer una versión más nueva, así que se crea un entorno aparte con `uv`. Toma unos minutos y hay que repetirlo en cada sesión nueva.

In [ ]:
%%bash
pip install -q uv
cd /content/r2dreamer-cookie
uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -e ".[cookie]" 2>&1 | tail -3

## 5. Comprobar que todo quedó bien

In [ ]:
%%bash
cd /content/r2dreamer-cookie
.venv/bin/python - <<'EOF'
import torch
from envs.cookie import Cookie
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")
a = torch.randn(256, 256, device="cuda", dtype=torch.float16)
print("float16 ok:", bool(torch.isfinite((a @ a).sum())))
env = Cookie("full", size=(80, 80))
obs = env.reset()
print("galleta ok:", obs["image"].shape, "| acciones:", env.action_space.n, "(deberían ser 3)")
print("métricas nuevas:", sorted(k for k in obs if k.startswith("log_")))
EOF

## 6. Configurar el run

**No cambies `TAREA`, `PASOS`, `TRAIN_RATIO`, `IMAGEN`, `REP_LOSS` ni `BUFFER` a mitad del run:** el checkpoint quedó hecho con esos valores y al retomar no calzaría.

- `GUARDAR_BUFFER_CADA`: el modelo se guarda cada 10.000 pasos, pero el buffer (~30 GB al final) solo cada 100.000, porque cada guardado lo escribe completo. Si la sesión se corta sola, se pierden como mucho los datos juntados desde el último guardado del buffer, y se vuelven a juntar. Si lo detienes tú (celda 10), se guarda todo.
- `PARAR_EN_PASO`: para que se detenga solo en un paso y guarde todo, por ejemplo `500_000` para dejarlo de noche. Después se sigue con `None` o con un paso mayor.
- `APAGAR_AL_TERMINAR`: con `True`, la sesión se apaga sola al terminar o al llegar a `PARAR_EN_PASO`, y deja de gastar unidades. No se apaga si lo detienes tú.

In [ ]:
CASO = "base"  # Dreamer normal: el actor junta los datos

TAREA = "cookie_full"  # visión completa; apretar el botón cambia la galleta de esquina
PASOS = 1_000_000
TRAIN_RATIO = 256
IMAGEN = 80  # píxeles por lado
REP_LOSS = "dreamer"  # con decoder
SEMILLA = 0
BUFFER = 1_000_000  # todo el run, en la RAM
GUARDAR_BUFFER_CADA = 100_000
EVALUAR_CADA = 50_000
PARTIDAS_EVAL = 5

# Para que se detenga solo en un paso y siga otro día, por ejemplo 500_000. None: hasta el final.
PARAR_EN_PASO = None
# Apaga la sesión cuando el entrenamiento termina o llega a PARAR_EN_PASO.
APAGAR_AL_TERMINAR = True

LOGDIR = f"/content/drive/MyDrive/galleta/semana8/{TAREA}_s{SEMILLA}_{CASO}_nano"

OPCIONES = (
    f"env=cookie env.task={TAREA} env.train_ratio={TRAIN_RATIO} 'env.size=[{IMAGEN},{IMAGEN}]' "
    f"seed={SEMILLA} model.rep_loss={REP_LOSS} trainer.random_policy={CASO == 'random'} "
    f"trainer.video_pred_log=True buffer.max_size={BUFFER} buffer.storage_device=cpu"
)
print(LOGDIR)
print(OPCIONES)

## 7. Ver en qué va el run

Dice en qué paso quedó el run guardado en Drive, desde qué paso es el buffer y cómo vienen las últimas partidas comparadas con un agente al azar. Córrela antes de entrenar en una sesión nueva para saber desde dónde va a seguir. Si dice que todavía no hay nada guardado, es un run nuevo.

In [ ]:
!cd /content/r2dreamer-cookie && python3 progress.py --logdir "{LOGDIR}"

## 8. Prueba corta (solo la primera vez)

Corre 1.500 pasos en el disco local con la misma configuración, incluido el buffer de 1.000.000 en la RAM, pero sin guardarlo. Comprueba que todo funcione y estima cuánto va a tardar el run. Toma unos 10 a 15 minutos.

In [ ]:
!rm -rf /content/prueba_nano
!cd /content/r2dreamer-cookie && timeout 1800 .venv/bin/python -u train.py {OPCIONES} \
    env.steps=1500 env.eval_episode_num=1 trainer.update_log_every=500 trainer.save_every=1e9 \
    trainer.save_buffer=False logdir=/content/prueba_nano > /content/prueba_nano.log 2>&1; \
    tail -n 5 /content/prueba_nano.log

In [ ]:
import json

lineas = [json.loads(l) for l in open("/content/prueba_nano/metrics.jsonl")]
fps = [l["fps/fps"] for l in lineas if l.get("fps/fps", 0) > 0]
print("evaluación ok:", any("episode/eval_score" in l for l in lineas))
print("métricas nuevas ok:", any("episode/eval_cells" in l for l in lineas))
if not fps:
    print("No alcanzó a medir la velocidad. Revisa el final de /content/prueba_nano.log (arriba).")
else:
    v = sum(fps) / len(fps)
    horas = PASOS / v / 3600
    print(f"{v:.1f} pasos por segundo → {PASOS:,} pasos en unas {horas:.0f} horas "
          f"(~{1.54 * horas:.0f} unidades en una L4), más los guardados del buffer y las evaluaciones")

## 9. Abrir TensorBoard

Ábrelo **antes** de entrenar: mientras la celda de entrenamiento corre, Colab no deja ejecutar otras, pero TensorBoard se sigue actualizando (botón de recargar arriba a la derecha).

En la pestaña **Scalars** (entre paréntesis, lo que hace un agente al azar):

- `episode/score`: galletas por partida mientras entrena (~0,02). Es lo que graficaba Nano.
- `episode/presses`: veces que aprieta el botón por partida (~7; el azar ni lo toca en el 60 % de las partidas).
- `episode/cells`: casillas distintas que visita por partida (~51).
- `episode/button_room`: pasos que pasa en el cuarto del botón (~450).
- `episode/eval_score`: galletas por partida del actor sin azar, cada 50.000 pasos.
- `train/rew`: la recompensa que predice el modelo al soñar. Si se queda en ~0,00001, el modelo cree que nunca hay galleta.
- `train/action_entropy`: 1,099 es azar puro; si baja, el actor empezó a preferir algo.

En la pestaña **Images**:

- `train_video`: la última partida de entrenamiento completa.
- `open_loop`: arriba lo real, al medio lo que predice el modelo y abajo la diferencia. Los primeros 5 cuadros los reconstruye mirando la imagen; el resto los **imagina** solo con las acciones. Si el modelo entendió la galleta, al imaginar la hace aparecer después de que el agente aprieta el botón.

In [ ]:
%load_ext tensorboard
from tensorboard import notebook

notebook.start(f'--logdir "{LOGDIR}"')

## 10. Entrenar (o seguir)

Arriba aparece una barra de progreso con el tiempo que falta, y debajo va corriendo el log. Al retomar, cargar el buffer desde Drive tarda unos minutos antes de que la barra aparezca.

**Para detenerlo:** usa el botón de detener de esta celda. El entrenamiento termina el paso en que va, guarda el modelo y el buffer en Drive (puede tardar varios minutos) y avisa con `Listo, quedó guardado`. No cierres la pestaña antes de ese mensaje.

**Para seguir otro día:** corre las celdas 1 a 6 y después esta. Sigue exactamente desde el paso en que lo detuviste.

**Si la sesión se corta sola** (desconexión, límite de Colab): sigue igual, desde el último guardado. El modelo se guarda cada 10.000 pasos y el buffer cada 100.000; el log avisa qué tramo de datos se perdió, y se vuelve a juntar.

Para dejarlo corriendo de noche: deja la pestaña abierta y que el PC no entre en suspensión (la pantalla sí se puede apagar). Cada guardado reemplaza el buffer anterior, que queda en la papelera de Drive; puedes vaciarla al final.

In [ ]:
import re
import subprocess

from tqdm.auto import tqdm

parar = f" trainer.stop_at={PARAR_EN_PASO}" if PARAR_EN_PASO else ""
comando = (
    # exec deja a Python en lugar de la shell, para que el botón de detener le llegue a él.
    f"exec .venv/bin/python -u train.py {OPCIONES} env.steps={PASOS} env.eval_episode_num={PARTIDAS_EVAL} "
    f"trainer.eval_every={EVALUAR_CADA} trainer.update_log_every=2500 trainer.save_every=1e4 "
    f'trainer.save_buffer_every={GUARDAR_BUFFER_CADA}{parar} logdir="{LOGDIR}"'
)
proceso = subprocess.Popen(comando, shell=True, cwd="/content/r2dreamer-cookie", text=True, bufsize=1,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
barra = None
detenido = False


def mostrar(linea):
    global barra
    paso = re.match(r"\[(\d+)\]", linea)
    if paso:
        paso = min(int(paso.group(1)), PASOS)
        if barra is None:
            # Parte desde el paso en que va, por si el run se retomó a mitad de camino.
            barra = tqdm(total=PASOS, initial=paso, unit="paso", smoothing=0, desc=CASO)
        barra.update(paso - barra.n)
        if "eval_score" in linea or "Saved the replay buffer" in linea:
            barra.set_postfix_str("")
        if " progress " in linea:
            return  # la barra ya muestra el avance
    if barra is not None and linea.startswith("Evaluating"):
        barra.set_postfix_str("evaluando")
    if barra is not None and "Saving the replay buffer" in linea:
        barra.set_postfix_str("guardando el buffer")
    print(linea, end="")


try:
    for linea in proceso.stdout:
        mostrar(linea)
except KeyboardInterrupt:
    detenido = True
    # train.py recibe la señal, termina el paso en que va y guarda todo antes de salir.
    proceso.terminate()
    print("\nDeteniendo: se está guardando el modelo y el buffer en Drive. Puede tardar varios minutos; "
          "no cierres la pestaña hasta que diga que quedó guardado.")
    while proceso.poll() is None:
        try:
            for linea in proceso.stdout:
                print(linea, end="")
        except KeyboardInterrupt:
            print("Ya se está guardando; espera a que termine.")
finally:
    if barra is not None:
        barra.close()
proceso.wait()
if proceso.returncode != 0:
    print(f"El entrenamiento terminó con un error (código {proceso.returncode}). Revisa las últimas líneas de arriba.")
elif detenido:
    print("Listo, quedó guardado. Para seguir otro día: celdas 1 a 6 y después esta.")

if APAGAR_AL_TERMINAR and not detenido and proceso.returncode == 0:
    from google.colab import drive, runtime

    # Drive sube los archivos con atraso: esto espera a que todo quede guardado antes de apagar.
    print("Guardando todo en Drive y apagando la sesión para que deje de gastar unidades...")
    drive.flush_and_unmount()
    runtime.unassign()

## 11. Ver el progreso

Lo mismo que la celda 7, más los gráficos: galletas, veces que aprieta el botón, casillas visitadas, pasos en el cuarto del botón, la recompensa que predice el modelo y la entropía del actor. La línea gris es un agente al azar y la verde punteada, el paso en que el run de Nano empezó a aprender.

Mientras la celda 10 corre, esto mismo se ve desde `monitor.ipynb` en otra pestaña.

In [ ]:
!cd /content/r2dreamer-cookie && python3 progress.py --logdir "{LOGDIR}" --out /content/progreso.png

from IPython.display import Image, display

display(Image("/content/progreso.png"))

## 12. Ver al agente jugar

Carga el agente desde Drive (`latest.pt`, el último guardado) y lo pone a jugar una partida completa con lo que aprendió, sin azar, igual que en la evaluación. El video queda en la carpeta del run.

- A la izquierda está el mapa y a la derecha la imagen que recibe el agente.
- Arriba van el paso, las galletas que lleva y la jugada que eligió.

Con `SEMILLA_EPISODIO` cambian las esquinas donde aparece la galleta. Tarda uno o dos minutos.

In [ ]:
SEMILLA_EPISODIO = 0
VIDEO = f"{LOGDIR}/agente_episodio{SEMILLA_EPISODIO}.mp4"

!cd /content/r2dreamer-cookie && .venv/bin/python watch.py --logdir "{LOGDIR}" --out "{VIDEO}" \
    --seed {SEMILLA_EPISODIO} 2>&1 | tail -n 1

from IPython.display import Video
Video(VIDEO, embed=True, width=760)